# 065 — Clasificación, extracción y generación de texto

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Clasificación:** `texto → TF-IDF → clasificador lineal` sigue siendo un baseline serio.
`tfidf(t,d) = tf(t,d) · log(N/df(t))`: términos frecuentes en el documento pero raros en
el corpus pesan más; "el" (df = N) pesa 0. Evaluar con F1 **por clase** (macro/micro), no
con accuracy global si hay desbalance.

**Extracción (NER):** etiquetado token a token con esquema **BIO** (`B-PER I-PER O …`);
la métrica es F1 **por entidad completa** (span + tipo exactos).

**Generación:** un modelo de lenguaje factoriza `P(w1…wn) = Π P(wi | contexto)` y genera
muestreando. **Perplejidad** `PP = P(secuencia)^(-1/n)` = factor de ramificación efectivo
(PP 4 → duda entre ~4 opciones por token); menor es mejor, solo comparable con mismo
tokenizador y corpus. Fluidez ≠ veracidad.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** df: `ia=2, genera=2, texto=2, clasifica=1, el=1, modelo=1, codigo=1`.
idf: `ia = genera = texto = ln(3/2) ≈ 0.405`; `clasifica = el = modelo = codigo =
ln(3/1) ≈ 1.099`. Vector de d1 (todos con tf = 1):
`ia → 0.405, genera → 0.405, texto → 0.405`. Los tres empatan; ninguno alcanza 1.099
porque todos aparecen en 2 de 3 documentos. En d2, `clasifica` pesaría 1.099: es el
término más distintivo del corpus.

**Ejercicio 2.**
`Pedro=B-PER, Pascal=I-PER, visitó=O, la=O, sede=O, de=O, Naciones=B-ORG, Unidas=I-ORG,
en=O, Nueva=B-LOC, York=I-LOC` → **3 entidades** (PER, ORG, LOC). En F1 por entidad NO hay
acierto parcial: `Naciones` sin `Unidas` falla el span completo → cuenta como FP + FN.

**Ejercicio 3.** Modelo 1: `P = 10^-3`, `PP = (10^-3)^(-1/3) = 10`. Modelo 2:
`P = 0.5 · 0.02 · 0.5 = 0.005`, `PP = 0.005^(-1/3) ≈ 5.85`. El modelo 2 duda menos en
promedio pese al token de 0.02: la perplejidad es una **media geométrica** inversa, un
token muy improbable la castiga, pero dos tokens de 0.5 lo compensan parcialmente.

**Ejercicio 4.** Implementación debajo; reproduce los pesos calculados a mano.


In [ ]:
result = run_lab("llm", seed=65)
assert result["kind"] == "llm"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 4 — TF-IDF verificado con código
import math
from collections import Counter

corpus = ["ia genera texto", "ia clasifica texto", "el modelo genera codigo"]

def tfidf(corpus):
    docs = [d.split() for d in corpus]
    N = len(docs)
    vocab = sorted({t for d in docs for t in d})
    df = {t: sum(t in d for d in docs) for t in vocab}
    idf = {t: math.log(N / df[t]) for t in vocab}
    vectores = []
    for d in docs:
        tf = Counter(d)
        vectores.append({t: round(tf[t] * idf[t], 3) for t in tf})
    return vectores

for i, v in enumerate(tfidf(corpus), 1):
    print(f"d{i}:", v)


In [ ]:
# Ejercicio 3 — perplejidad verificada con código
def perplejidad(probs):
    p = 1.0
    for x in probs:
        p *= x
    return p ** (-1 / len(probs))

print(round(perplejidad([0.1, 0.1, 0.1]), 2))    # 10.0
print(round(perplejidad([0.5, 0.02, 0.5]), 2))   # ~5.85
print(round(perplejidad([0.5, 0.25, 0.5, 0.125]), 2))  # 4.0 (ejemplo de la teoría)


## Reflexión

1. Tu clasificador de spam tiene accuracy 97 % pero recall de spam 40 %. ¿Qué pasó, qué
   métrica debiste vigilar y qué costo de error hace grave este caso?
2. En NER clínico, ¿por qué "F1 por token 92 %" puede convivir con extracciones inservibles,
   y qué exige la métrica por entidad que la hace más honesta?
3. Un modelo A tiene perplejidad 12 y un modelo B tiene 15 con tokenizadores distintos.
   ¿Puedes concluir que A genera mejor? ¿Qué experimento sí lo permitiría?
